# 01 — Minimal Decision Trace

This notebook runs one task through the procedural interpretability flow with and without the checklist. Requires `OPENAI_API_KEY` in `.env`.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from procedural_interp.io import load_jsonl
from procedural_interp.checklists import load_checklist, checklist_to_prompt
from procedural_interp.models import OpenAIChatModel
from procedural_interp.runner import run_one, select_context

checklist = load_checklist("../checklists/core_mmm_v0.1.yaml")
checklist_prompt = checklist_to_prompt(checklist)
tasks = load_jsonl("../data/tasks/sample_tasks.jsonl")
contexts = load_jsonl("../data/contexts/sample_contexts.jsonl")

In [ ]:
# Choose a task and context condition
task = tasks[0]
context_condition = "problematic"
context = select_context(contexts, task["task_id"], context_condition)
print(task["user_request"])
print("--- context ---")
print(context)

In [ ]:
model = OpenAIChatModel(model="gpt-4.1-mini", temperature=0.0)

trace_no_checklist = run_one(
    task=task,
    context=context,
    model_condition="clean",
    context_condition=context_condition,
    checklist_enabled=False,
    model=model,
    checklist_prompt=None,
)
trace_no_checklist.model_dump()

In [ ]:
trace_with_checklist = run_one(
    task=task,
    context=context,
    model_condition="clean",
    context_condition=context_condition,
    checklist_enabled=True,
    model=model,
    checklist_prompt=checklist_prompt,
)
trace_with_checklist.model_dump()

## What to inspect

- Did the final action change?
- Did the checklist identify stale/misleading context?
- Did the final action follow from the checklist results?
- Did the deterministic consistency check flag anything?